In [ ]:
import numpy as np
import mne
import os
from pactools import Comodulogram
from tqdm import tqdm
import warnings
import scipy.io as sio

warnings.filterwarnings('ignore')

# ====================== 1. Global Parameters ======================
SFREQ = 500

PHASE_FREQS = np.linspace(0.5, 20, 39)
AMP_FREQS = np.linspace(30, 90, 60)

N_SURROGATES = 200

# ====================== 2. Channel Definition ======================
# Anterior channels (non-occipital, first 23 channels)
ANTERIOR_CHANNELS = [
    'Fpz','Fp2','Fz','F4','F8',
    'FC1','FC2','FC6',
    'M1','C3','Cz','C4','T8','M2',
    'CP5','CP1','CP2',
    'P7','P3','Pz','P4','P8','POz'
]

# Occipital channels
OCCIPITAL_CHANNELS = ['O1', 'Oz', 'O2']

# All channels to be loaded
ALL_CHANNELS = list(set(ANTERIOR_CHANNELS + OCCIPITAL_CHANNELS))

# ====================== 3. Bidirectional PAC Configuration ======================
PAC_DIRECTIONS = {
    'top_down': {
        'phase': ANTERIOR_CHANNELS,
        'amp': OCCIPITAL_CHANNELS
    },
    'bottom_up': {
        'phase': OCCIPITAL_CHANNELS,
        'amp': ANTERIOR_CHANNELS
    }
}

# ====================== 4. Data Paths ======================
GROUPS = {
    'sham': {
        'subjects': ["300207", "300308", "300409", "300618", "300720","300822", "300925", "301041", 
                     "301250", "301354", "301455", "301560", "301662", "301766", "301867", "302177", 
                     "302282", "302384"],
        'source_dir': r'D:\山东第一医科大学\数据\TI_TASK_DATA\Task2\八因子\预处理\SHAM'
    }
}

OUT_BASE_DIR = r'D:\山东第一医科大学\数据\Task\八因子\频段耦合\跨被试\pac\其他调制枕叶'

TIME_POINTS = ['post']
EMOTIONS = ['sad']

# ====================== 5. Single-Subject PAC Calculation Function ======================
def calculate_pac_for_subject(subject, source_dir, out_dir):

    factor_results = []

    for factor in range(1, 9):

        fif_path = os.path.join(
            source_dir, TIME_POINTS[0], EMOTIONS[0],
            subject, f'factor_{factor}.fif'
        )

        raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)

        data = raw.get_data(picks=ALL_CHANNELS)
        data = mne.filter.detrend(data, axis=1)
        data -= data.mean(axis=1, keepdims=True)

        ch_to_idx = {ch: i for i, ch in enumerate(ALL_CHANNELS)}

        factor_dir_results = {}

        # ====================== Bidirectional PAC ======================
        for direction, cfg in PAC_DIRECTIONS.items():

            phase_chs = cfg['phase']
            amp_chs = cfg['amp']

            n_phase = len(phase_chs)
            n_amp = len(amp_chs)

            cross_pac = np.zeros((n_phase, n_amp, len(AMP_FREQS), len(PHASE_FREQS)))
            cross_z = np.zeros_like(cross_pac)
            cross_m = np.zeros((n_phase, n_amp, N_SURROGATES))

            comod = Comodulogram(
                fs=SFREQ,
                low_fq_range=PHASE_FREQS,
                high_fq_range=AMP_FREQS,
                method='tort',
                n_surrogates=N_SURROGATES,
                progress_bar=False,
                n_jobs=10
            )

            for i_p, ch_p in enumerate(phase_chs):
                for i_a, ch_a in enumerate(amp_chs):

                    print(
                        f'Subject {subject} | Factor {factor} | '
                        f'{direction} | Phase: {ch_p} → Amp: {ch_a}'
                    )

                    comod.fit(
                        data[ch_to_idx[ch_p]],
                        data[ch_to_idx[ch_a]]
                    )

                    cross_pac[i_p, i_a] = comod.comod_.T
                    cross_z[i_p, i_a] = comod.comod_z_score_.T
                    cross_m[i_p, i_a] = comod.surrogate_max_

            factor_dir_results[direction] = {
                'pac': cross_pac,
                'pac_z': cross_z,
                'surrogate_max': cross_m,
                'phase_channels': phase_chs,
                'amp_channels': amp_chs
            }

        factor_results.append(factor_dir_results)

    # ====================== Save Results ======================
    save_dict = {
        'results': factor_results,
        'freqs': {
            'phase_freqs': PHASE_FREQS,
            'amp_freqs': AMP_FREQS
        }
    }

    out_path = os.path.join(out_dir, 'pac_bidir.mat')
    sio.savemat(out_path, save_dict)
    print(f'Subject {subject} bidirectional PAC saved successfully: {out_path}')

# ====================== 6. Batch Processing ======================
if __name__ == '__main__':

    for group_name, group_info in GROUPS.items():
        src_dir = group_info['source_dir']
        subjects = group_info['subjects']

        for time_point in TIME_POINTS:
            for emotion in EMOTIONS:
                for subject in tqdm(subjects, desc=f'{group_name}-{time_point}-{emotion}'):

                    out_dir = os.path.join(
                        OUT_BASE_DIR, group_name, time_point, emotion, subject
                    )
                    os.makedirs(out_dir, exist_ok=True)

                    calculate_pac_for_subject(
                        subject=subject,
                        source_dir=src_dir,
                        out_dir=out_dir
                    )

    print('✅ All subjects: Bidirectional PAC calculation completed')

In [ ]:
# ===================== Save Completion Prompt =====================
print(outpath, "saved successfully")
print(f"PAC raw value array shape: {np.array(factor_all_pac, dtype=object).shape}")
print(f"PAC z-score array shape: {np.array(factor_all_z, dtype=object).shape}")
print(f"Surrogate max array shape: {np.array(factor_all_m, dtype=object).shape}")